# 原39项联合SOXX优势：精确f696母本的完整0bp账户

先验证944行f696五输出身份、训练模型与决策时间，再按固定0.10/0.10转移规则运行候选自身的b472动作、风险、整股、费用与NAV。SOXX为硬门。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from src.workflow.stage1_nvda_guidance_receiver import prepare_account_sources

root=Path.cwd();card=root/"research/stage1_joint_soxx_bridge_0bp_20261024"
plan_path=card/"plan.json";plan=json.loads(plan_path.read_text());src=plan["source"]
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and plan["execution_authorized_by_active_Goal"] and plan["economics"]["added_slippage_bps"]==0
for name,digest in src["code_sha256"].items():assert sha(root/name)==digest,name
info=root/src["information_card"];f696=root/src["exact_f696_card"]
for file,key in [(info/"plan.json","information_plan_sha256"),(info/"result.json","information_result_sha256"),(info/"summary.json","information_summary_sha256"),(f696/"summary.json","exact_f696_summary_sha256")]:assert sha(file)==src[key],key
ir=json.loads((info/"result.json").read_text());ii=json.loads((info/"summary.json").read_text())
assert all(ir["gates"].values()) and ir["NAS_readback_verified"] and ii["NAS_readback_verified"]
assert ii["NAS_evidence"]["sha256"]==src["information_NAS_sha256"] and ii["NAS_evidence"]["size_bytes"]==src["information_NAS_size_bytes"]
info_art=info/"qlib_recorder"/src["information_experiment_id"]/src["information_recorder_id"]/"artifacts"
assert sha(info_art/"models/joint_margin")==src["model_artifact_sha256"] and sha(info_art/"inputs/dataset")==src["dataset_artifact_sha256"]
assert sha(root/src["original_timing_artifact"])==src["original_timing_sha256"]
assert sha(root/src["provider_uri"]/"qlib_store_manifest.json")==src["provider_manifest_sha256"]
fart=f696/"qlib_recorder"/src["exact_f696_experiment_id"]/src["exact_f696_recorder_id"]/"artifacts/inputs/frame"
assert sha(fart)==src["exact_f696_frame_sha256"]
helper=root/src["account_source_helper"];receipt_path=helper/"account_source_preflight.json"
assert sha(receipt_path)==src["account_source_receipt_sha256"]
account=prepare_account_sources(root=root,card=helper)
assert sha(receipt_path)==src["account_source_receipt_sha256"]
assert len(account["frame"])==plan["economics"]["sessions"]==944 and len(account["points"])==plan["economics"]["price_marks"]==3778
assert account["references"]["Stage1_A"]["audit"]["status"]==account["references"]["SOXX"]["audit"]["status"]=="passed"
with R.uri_context(str((f696/"qlib_recorder").resolve())):
 f696_rec=R.get_recorder(experiment_id=src["exact_f696_experiment_id"],recorder_id=src["exact_f696_recorder_id"])
 f696_frame=f696_rec.load_object("inputs/frame").sort_values("trade_date").reset_index(drop=True)
with R.uri_context(str((info/"qlib_recorder").resolve())):
 info_rec=R.get_recorder(experiment_id=src["information_experiment_id"],recorder_id=src["information_recorder_id"])
 model=info_rec.load_object("models/joint_margin")
 dataset=info_rec.load_object("inputs/dataset")
held=dataset.prepare("held",col_set=["feature","meta"])
dates=pd.DatetimeIndex(held.index.get_level_values("datetime"))
frame=account["frame"].sort_values("trade_date").reset_index(drop=True).copy(deep=True)
assert len(held)==len(frame)==len(f696_frame)==944 and dates.equals(pd.DatetimeIndex(frame.signal_date))
assert pd.DatetimeIndex(f696_frame.signal_date).equals(dates)
source_targets=account["source_targets"]
columns=["soxl_weight","soxx_weight","cash_weight","tail_probability","alpha_probability"]
assert list(source_targets.columns)==columns and source_targets.index.equals(dates)
for name in columns:frame[name]=source_targets[name].to_numpy(float)
parity={name:float(np.max(np.abs(frame[name].to_numpy(float)-f696_frame[name].to_numpy(float)))) for name in columns}
assert max(parity.values())==0,parity
with R.uri_context(str((root/"research/minute_information_pit_allocator_20261012/qlib_recorder").resolve())):
 timing_rec=R.get_recorder(experiment_id="905520356740140469",recorder_id="68e31a0a0fad44e7ba1d1f7d856c7a0d")
 timing=timing_rec.load_object("inputs/timing")
decisions=pd.to_datetime(timing.reindex(dates).decision_at,utc=True).dt.tz_convert("America/New_York").dt.date
assert all(d>x.date() for d,x in zip(decisions,dates))
pred=np.asarray(model.predict(held["feature"]),float)
assert pred.shape==(944,) and np.isfinite(pred).all()
trigger=pred>0
candidate=frame.copy(deep=True)
old_soxl=frame.soxl_weight.to_numpy(float);old_cash=frame.cash_weight.to_numpy(float)
move_soxl=np.where(trigger,np.minimum(0.10,np.maximum(old_soxl,0)),0.0)
move_cash=np.where(trigger,np.minimum(0.10,np.maximum(old_cash-0.15,0)),0.0)
candidate["soxl_weight"]=old_soxl-move_soxl
candidate["cash_weight"]=old_cash-move_cash
candidate["soxx_weight"]=frame.soxx_weight.to_numpy(float)+move_soxl+move_cash
assert np.isfinite(candidate[columns].to_numpy(float)).all()
assert float(candidate[["soxl_weight","soxx_weight","cash_weight"]].min().min())>=-1e-12
assert float((candidate[["soxl_weight","soxx_weight","cash_weight"]].sum(axis=1)-1).abs().max())<1e-12
assert np.allclose(candidate.loc[~trigger,columns],frame.loc[~trigger,columns],atol=0,rtol=0)
assert np.array_equal(candidate[["tail_probability","alpha_probability"]].to_numpy(),frame[["tail_probability","alpha_probability"]].to_numpy())
assert np.all(move_cash<=0.10+1e-12) and np.all(move_soxl<=0.10+1e-12)
preflight={"status":"exact_f696_parent_and_joint_model_full_calendar_mapping_pass","plan_sha256":sha(plan_path),
           "source_model_recorder_id":src["information_recorder_id"],"source_account_receipt_sha256":src["account_source_receipt_sha256"],
           "f696_five_output_max_abs":parity,"sessions":len(candidate),"source_points":len(account["points"]),
           "trigger_days":int(trigger.sum()),"cash_transfer_days":int((move_cash>0).sum()),"SOXL_transfer_days":int((move_soxl>0).sum()),
           "mean_cash_transfer":float(move_cash.mean()),"mean_SOXL_transfer":float(move_soxl.mean()),
           "new_model_fits":0,"new_native_accounts":0}
(card/"preflight_result.json").write_text(json.dumps(preflight,ensure_ascii=False,indent=2)+"\n")
print({k:preflight[k] for k in ["status","sessions","trigger_days","cash_transfer_days","SOXL_transfer_days","mean_cash_transfer","mean_SOXL_transfer"]})


{'status': 'exact_f696_parent_and_joint_model_full_calendar_mapping_pass', 'sessions': 944, 'trigger_days': 0, 'cash_transfer_days': 0, 'SOXL_transfer_days': 0, 'mean_cash_transfer': 0.0, 'mean_SOXL_transfer': 0.0}


In [2]:
from src.backtest.stage1_domain_heads_raw import replay_stage1_raw
from src.backtest.native_raw_corporate import native_raw_validate
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
from src.workflow.stage1_rolling126_shortfall import _analyze_account
from src.data.nas_market_data import archive_nas_recorder,materialize_nas_evidence
import tempfile
assert preflight["status"]=="exact_f696_parent_and_joint_model_full_calendar_mapping_pass"
state_path=card/"run_state.json"
if state_path.exists():raise ValueError("fixed full Stage1 joint-bridge account already attempted; do not replay")
state={"status":"native_account_attempted","plan_sha256":sha(plan_path),"new_model_fits":0,"native_account_attempts":1,"bootstrap_attempts":0}
state_path.write_text(json.dumps(state,indent=2)+"\n")
recovery=IntradayRecoveryDirectAllocationSpec(root=root,**account["recovery_spec"])
action=Stage1OpenActionValueSpec(root=root,**account["action_spec"])
execution={"initial_cash":plan["economics"]["initial_cash_usd"],"tariff":account["parent"]["tariff"]}
path=replay_stage1_raw(candidate,point_prices=account["points"],schedule=account["schedule"],plan=execution,
                       controller=account["controller"],recovery_spec=recovery,action_spec=action,
                       selection=account["selection"],slippage_bps=0)
assert len(path["replay"])==944 and len(path["points"])==3778
book=native_raw_validate(path,provider_uri=root/plan["source"]["provider_uri"],point_prices=account["points"],
                         schedule=account["schedule"],tariff=execution["tariff"],cost=0,
                         initial_cash=execution["initial_cash"])
assert book["audit"]["status"]=="passed"
state["status"]="native_complete";state["native_account_completed"]=1;state_path.write_text(json.dumps(state,indent=2)+"\n")
cols=["signal_date","trade_date","exit_date"]
for role in ("Stage1_A","SOXX"):
 pd.testing.assert_frame_equal(book["authority"][cols].reset_index(drop=True),account["references"][role]["authority"][cols].reset_index(drop=True))
state["bootstrap_attempts"]=1;state_path.write_text(json.dumps(state,indent=2)+"\n")
analysis=_analyze_account(account=book["authority"],reference_soxx=account["references"]["SOXX"]["authority"],
                          reference_stage1=account["references"]["Stage1_A"]["authority"],
                          sampled_drawdown=-100*float(path["replay"].monitored_drawdown.max()),plan=plan)
with R.start(experiment_name=card.name,uri=str((card/"qlib_recorder").resolve())):
 recorder=R.get_recorder()
 recorder.save_objects(artifact_path="inputs",plan=plan,preflight=preflight,source_receipt=account["receipt"],candidate_frame=candidate)
 recorder.save_objects(artifact_path="predictions",joint_margin_full=pd.Series(pred,index=dates),trigger=pd.Series(trigger,index=dates))
 recorder.save_objects(artifact_path="paths",candidate_0=path)
 recorder.save_objects(artifact_path="native",candidate_0=book)
 recorder.save_objects(artifact_path="analysis",SOXX_metrics=analysis["SOXX_metrics"],Stage1_increment=analysis["Stage1_increment_metrics"],
                       annual=analysis["annual"],quarter=analysis["quarter"],rolling_summary=analysis["rolling_summary"],
                       uncertainty=analysis["uncertainty"],gates=analysis["gates"],mapping_audit=preflight)
 recorder.save_objects(artifact_path="analysis/rolling",all_windows=analysis["rolling"])
 R.log_params(parent="exact_f696_A0",receiver="direct_original39_joint_SOXX_margin",added_slippage_bps=0)
 R.log_metrics(native_accounts=1,SOXX_relative_wealth=analysis["SOXX_metrics"]["relative_wealth"],
               historical_0bp_pass=int(analysis["historical_0bp_gate_pass"]))
 rid,eid=recorder.id,str(recorder.experiment_id)
state.update(status="account_and_analysis_complete",recorder_id=rid,experiment_id=eid)
state_path.write_text(json.dumps(state,indent=2)+"\n")
parent_binding=json.loads((root/"research/stage1_domain_heads_20261018/nas_binding.json").read_text())
binding={"schema_version":"joint_SOXX_bridge_0bp_recorder_backup_v1","authority":"yanxuan_nas","ssh":parent_binding["ssh"],
         "source_id":"exact_f696_joint_bridge_"+sha(plan_path)[:16]}
binding_path=card/"nas_binding.json";binding_path.write_text(json.dumps(binding,indent=2)+"\n")
nas=None;nas_error=None;readback=False
try:
 nas=archive_nas_recorder(binding_path,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
 with tempfile.TemporaryDirectory(prefix="joint-bridge-account-readback-") as temp:
  restored=Path(temp);materialize_nas_evidence(binding_path,card=restored)
  with R.uri_context(str((restored/"qlib_recorder").resolve())):
   check=R.get_recorder(experiment_id=eid,recorder_id=rid)
   m=check.load_object("analysis/SOXX_metrics");g=check.load_object("analysis/gates")
  readback=m==analysis["SOXX_metrics"] and g==analysis["gates"]
except Exception as exc:nas_error=f"{type(exc).__name__}: {str(exc)[:300]}"
result={"status":"historical_0bp_pass_needs_independent_review" if analysis["historical_0bp_gate_pass"] and readback else
                 "not_promoted_fixed_joint_SOXX_bridge_0bp" if readback else "needs_NAS_account_readback",
        "objective_achieved":False,"plan_sha256":sha(plan_path),"preflight":"preflight_result.json",
        "SOXX_metrics":analysis["SOXX_metrics"],"Stage1_increment_metrics":analysis["Stage1_increment_metrics"],
        "rolling_summary":analysis["rolling_summary"],"uncertainty":analysis["uncertainty"],"gates":analysis["gates"],
        "historical_0bp_gate_pass":analysis["historical_0bp_gate_pass"],
        "annual":analysis["annual"].to_dict(orient="records"),"quarter":analysis["quarter"].to_dict(orient="records"),
        "recorder_id":rid,"experiment_id":eid,"NAS_evidence":nas,"NAS_readback_verified":readback,"NAS_error":nas_error,
        "new_model_fits":0,"new_native_accounts":1,"new_bootstraps":1,"independent_forward_days":0,
        "full_minute_risk_qualified":False,"SOXX_goal_achieved":False,
        "next_action":"If any historical0bp hard gate fails, close this exact mapping with one minimal attribution. If historical gates pass, request independent scoped result review before separately freezing2/5bp and qualified forward evidence."}
(card/"result.json").write_text(json.dumps(result,ensure_ascii=False,indent=2,default=str)+"\n")
state["status"]="completed" if readback else "needs_NAS_readback";state_path.write_text(json.dumps(state,indent=2)+"\n")
print({"status":result["status"],"SOXX_relative_wealth":result["SOXX_metrics"]["relative_wealth"],
       "rolling_positive":result["rolling_summary"]["positive_fraction"],"drawdown":result["SOXX_metrics"]["max_drawdown_pct"],
       "block_lower":result["uncertainty"]["daily_lower_bound"],"gates":result["gates"],"NAS_readback":readback})


{'status': 'not_promoted_fixed_joint_SOXX_bridge_0bp', 'SOXX_relative_wealth': 0.6860064369263443, 'rolling_positive': 0.31135531135531136, 'drawdown': -28.095726754031546, 'block_lower': -0.0011808202292724647, 'gates': {'SOXX_terminal_net_wealth': False, 'available_drawdown_30pct': True, 'CVaR5_no_worse_SOXX': True, 'complete_years_70pct': False, 'rolling_126_70pct': False, 'SOXX_paired_lower_positive': False, 'full_minute_risk_qualified': False, 'independent_forward_qualified': False}, 'NAS_readback': True}


## 中文结论

固定信息门通过后，只对保存的原39项联合H5模型预测设自然零门。先将账户助手返回的 **27baa** 原始现金账本中五个提案输出全部替换为冻结 `source_targets["frozen_current"]`，逐日核对 **944/944** 与原 f696 A0 的五输出误差为0，再按冻结规则：预测>0时最多从现金和SOXL各转0.10至SOXX，保持原b472开盘动作、风险投影、恢复、整股和Tiered费，独立运行完整**0bp额外滑点**原生Qlib账户。来源身份、3778价格点、决策前时间、法定权重与NAS读回均通过；0新模型拟合、1个完整候选账户、1次冻结区块检验。

失败机制很明确：联合标签为正仅 **79/3014** 个成熟训练日、**33/940** 个复用评价日；训练后的模型在全部944个资金决策日预测值范围 **[-0.11708,-0.00322]**，没有一次大于0。故现金与SOXL转移均为0，候选原生账户的22个数值字段、日期、费用和净值路径与冻结f696 A0逐日完全一致，最大绝对差0。旧两头预测组合的MSE被新接收器明显改进，**却没有可执行正分数**；条件性信息门不能代替动作支持门。

SOXX相对最终财富仍 **0.686006**，完整2024/2025年仅2024胜出，连续126日正超额 **255/819**，校正区块下界 **-0.00118082**。可用点回撤与CVaR5门通过，但SOXX财富、完整年度、滚动窗口和统计下界均失败；全分钟风险与独立前向仍未合格。因此决定 `not_promoted_fixed_joint_SOXX_bridge_0bp_no_action`。不把零门改成负数、不增转移规模、不挑年份重测；该固定版本关闭。后续若寻找新机制，须分清“现金何时输给SOXX”和“SOXL何时输给SOXX”的**各自动作价值**，并用真正可在决策前产生动作的信号检验，不能把稀有的“SOXX同时胜过两者”MSE改善当成SOXX策略成功。
